# Practical Block III: Grand Challenge
# BACH: Grand Challenge on Breast Cancer Histology Images
By: Niklas Long Schiefelbein

---

## General Challenge Information

This Jupyter Notebook is presenting the results and findings of the third and last exercise of the course DLMIA based on the BACH challenge. This exercise did only consider Part A of the challenge - the classification of H&E stained breast histology microscopy images in four classes: *Normal*, *Benign*, *In situ carcinoma* and *Invasive carcinoma*.

The dataset for this part of the challenge consists of 400 training images (originally there was another test set consisting of 100 images to which no access was available), in which the four classes are equally represented. The images were acquired in 2014, 2015 and 2017 using a Leica DM 2000 LED microscope and a Leica ICC50 HD camera and all patients are from the Porto and Castelo Branco regions (Portugal).

---

## Methodology and Approach in this Exercise

The overall approach consisted of orienting around what the challenge winner of Part A [(Chennamsetty et al. (2018))](https://link.springer.com/chapter/10.1007/978-3-319-93000-8_91) did. Particularly, they decided to implement a classification ensemble consisting of one ResNet101 model and two DenseNet161 models of which the latter two incorporate different normalisation schemes (one is normalised on ImageNet and the other on the BACH dataset). However, simply clining their approach, which lead them win this challenge, does not grant the pedagogical benefits that could be acquired otherwise. Instead, the exercise's goal consisted of analysing which data handling techniques affect the performance and how. With this objective in mind, a multi-step approach was choosen in which several configurations are tested stage-wise:

1. **Data Augmentation:** Firstly, the effect of augmentations were analysed. Four different sets of augmentation were developed in order to assess their effect on the classification performance. 
    - **No Augmentation:** This only includes resizing and normalisation which are necessary steps in order to alter the input samples to work with CNNs. This scheme is also used for validation and testing since those steps have to be performed on the original data but resized and normalised.
    - **Simple Geometric Augmentations:** Includes horizontal and vertical flipping and random rotation. Those augmentation techniques imitate different orientations when taking photos of the tissue under the microscope. Since there is no inherent information in how a microscopy image is oriented, those augmentations can be used easily in order to increase the variance of the dataset.
    - **Complex Geometric Augmentations:** Includes all of the above augmentations but extends the set by `ShiftScaleRotate` and `ElasticTransform`. The first is again a more aggressive form of the already implemented augmentations whereas the latter simlulates differently distributed stretches of tissue.
    - **Photometric Augmentations:** Again, this set includes all of the above methods but extends with `HueSaturationValue` and `RandomBrightnessContrast` to account for variability in the colourspace. This should simulate different stain manufacturers, “fresh” stains and older, more faded slides and scanner light intensities.
    
    We initially focused solely on geometric augmentations to avoid potential loss of critical colour information (due to missing domain knowledge). Later, based on [Marini et al. (2023)](https://www.sciencedirect.com/science/article/pii/S2153353922007830?via%3Dihub), we introduced colour space transformations (`HueSaturationValue` and `RandomBrightnessContrast`) using the safe boundaries defined in the study.

2. **Image Resolution:** The second analysis consisted of analysing the effect of doubling the input width and height each. Instead of 256x256, the images were resized to 512x512 while reducing the batch size from 32 to 8 in order to account for the increased GPU capacity utilisation. 

3. **Learning Rate:** The third stage of the analysis takes the best performing configurations and runs the training once again but exploring different learning rates: [0.0001, 0.0005, 0.001].

**Ensemble Run:** Finally, an ensemble run is conducted which combines a 5-fold cross validation of each model's prediction and soft voting to get a more reliable and model-independent classification. The ensemble architecture follows [(Chennamsetty et al.)](https://link.springer.com/chapter/10.1007/978-3-319-93000-8_91) but the present experiment leverages the concept of Bagging (Bootstrap Aggregating) combined with Cross-Validation to mitigate the variance inherent in training on small datasets.
Important to mention is that here, due to the integrated cross validation, it was chosen to apply a soft voting mechanism instead of a majority voting which was used in the original paper. The implemented system is demonstrated in Figure 1.

The dataset was first divided into a training pool (90% of the whole dataset) and a stratified hold-out test set (10%). The training pool was then used to conduct a 5-Fold Stratified Cross-Validation. For each of the three selected model configurations (ResNet101-ImageNet, DenseNet161-ImageNet, DenseNet161-BACH), a separate instance was trained on each of the 5 folds. This resulted in a total of 15 trained models (3 architectures x 5 folds), ensuring that every data point in the training pool was used for training at least 4 times.

For the final evaluation, the unseen hold-out test set was passed through all 15 models. We employed soft voting instead of simple majority voting. This means the Softmax probability vectors from all 15 models were averaged to produce a single confidence vector for the ensemble. The class with the highest average probability was selected as the final prediction. This approach allows the ensemble to capture nuances of model confidence, where a strong confident prediction can outweigh multiple weak, uncertain ones.

<center><img src="soft_voting_cross_val.png" alt="grand_ensemble_system" width="600"/><center>

<small>Figure 1: The implemented Grand Ensemble System including Cross Validation and Soft Voting.</small>
<br><small>Note: This diagram was generated by Gemini 3 based on what has been implemented.</small>

# TODO Still to mention:
- split structure
- absence of different augmentation techniques due to fear of corrupting the samples
- stratified split
- soft voting with bagging
- mention that reached performance is not as high as challenge winners but system is more robust towards differences in staining and imaging
- mention that benign was the worst in the challenge which my ensemble was able to classify to 100% correctly
- mention that little data
- mention that hold out test set was also used to define winning architectures per stage but not in this notebook
- mention that loss is not shown since its information is also in accuracy plots
- talk about the balanced dataset and that therfore any oversampling/undersampling was not needed

***
## Results and Findings

### Experiment 1: Augmentation

<img src="summary_accuracy_grid.png" alt="Accuracy Histories with different Augmentation Schemes" width="100%"/>

<small>Figure 2: Overview of Accuracy Histories for each Model for each Augmentation Scheme.</small>

Figure 2 showcases the evolution of training histories throughout applying different augmentation sets. What is evident and what was most influential towards the following analysis is the clear sign of overfitting in early stages when no augmentation is applied. All models were able to reach 95% training accuracy or higher already after the second epoch while the validation accuracy stagnates. Since balancing overfitting and underfitting is one of the biggest challenges of deep learning, and already considering the quite high performances in accuracy of the very first baselines, the common thread throughout this practical was significantly aligned towards increasing the training complexity (regularising) to mitigate overfitting and improve generalisation.

The increasing complexity of augmentations that are applied lead to less steep learning curves which signalise the effect of an increasing difficulty to just memorise the data. All trainings show signs of increased volatility, which could have been mitigated with longer training (but was chosen not to due to limited computational ressources).

Just by looking at the graphs and validation performances of the last epoch, one can determine Augmentation Scheme 2 to yield the best results for each model. However, even though we were hesitant towards applying photometric augmentations first, we decided to use the `augmentation_strength=3` in further experiments, including photometric adjustments, to account for the (theoretical) variability of different scanning conditions in a real-world scenario.

These experiments were run with a `batch_size=32`, `learning_rate=0.0001`, `num_epochs=15` and `img_size=[256, 256]`. 

***
### Experiment 2: Image Resolution

<img src="summary_resolution_accuracy_grid.png" alt="Accuracy Histories with different Resolutions" width="100%"/>
<small>Figure 3: Overview of Accuracy Histories for each Model for Input Resolution 256x256 and 512x512.</small>

Figure 3 compares the runs per model with a 256x256 input resolution against a 512x512 input resolution. It seems that the increased number of computations (due to doubling the width and height) leads to a regularising effect. The divergence between training and validation accuracy at the end of the training with 512x512 is less noticeable compared to the 256x256 run. The DenseNet model normalised on ImageNet appears to be more volatile in terms of training accuracy, but this is not evident in the other two models.

Interestingly, the final and highest achieved validation accuracy throughout the entire training process was consistently in the 512x512 runs. This suggests that incorporating more details can lead to better classification results in this task.

These experiments were run with a `batch_size=8`, `learning_rate=0.0001`, `num_epochs=15` and `img_size=[512, 512]`. 

***
### Experiment 3: Learning Rates

<img src="summary_lr_accuracy_grid.png" alt="Accuracy Histories with different Learning Rates" width="100%"/>
<small>Figure 4: Overview of Accuracy Histories for each Model for Learning Rates 0.0001, 0.0005 and 0.001.</small>

Figure 4 validates the challenge winner's finding that 0.0001 is the optimal learning rate among those tested. While the 0.0001 plots show roughly stable convergence, the higher rates (0.0005 and 0.001) exhibit significant volatility in validation accuracy. This instability indicates that the gradient steps are too large, causing the optimisation process to overshoot the local minima instead of settling into them.

These experiments were run with a `batch_size=8`, `learning_rate=[0.0001, 0.0005, 0.001]`, `num_epochs=15` and `img_size=[512, 512]`. 

***
### Experiment 4: The Final Ensemble Run with Cross Validation and Soft Voting

<img src="summary_aggregated_confusion_matrices.png" alt="Confusion Matrices per Model: All five folds aggregated" width="100%"/>
<small>Figure 5: Aggregated Confusion Matrices per Model in the Final Ensemble Evaluation.</small>

<div style="display: flex; justify-content:">
    <div style="width: 48%;">

| Model                      | Fold | Norm     | Accuracy |
| :------------------------- | :--: | :------- | :------: |
| **ResNet101 (ImageNet)**   |  0   | imagenet |  87.50%  |
| **ResNet101 (ImageNet)**   |  1   | imagenet |  90.00%  |
| **ResNet101 (ImageNet)**   |  2   | imagenet |  87.50%  |
| **ResNet101 (ImageNet)**   |  3   | imagenet |  87.50%  |
| **ResNet101 (ImageNet)**   |  4   | imagenet |  82.50%  |
| **DenseNet161 (ImageNet)** |  0   | imagenet |  87.50%  |
| **DenseNet161 (ImageNet)** |  1   | imagenet |  90.00%  |
| **DenseNet161 (ImageNet)** |  2   | imagenet |  95.00%  |
| **DenseNet161 (ImageNet)** |  3   | imagenet |  87.50%  |
| **DenseNet161 (ImageNet)** |  4   | imagenet |  95.00%  |
| **DenseNet161 (BACH)**     |  0   | bach     |  97.50%  |
| **DenseNet161 (BACH)**     |  1   | bach     |  90.00%  |
| **DenseNet161 (BACH)**     |  2   | bach     |  85.00%  |
| **DenseNet161 (BACH)**     |  3   | bach     |  95.00%  |
| **DenseNet161 (BACH)**     |  4   | bach     |  92.50%  |

<small>Table 1: Each Model Accuracy per Fold</small>

</div>
<div style="width: 48%;">


| Model Configuration        | Mean Accuracy | Std Dev |
| :------------------------- | :-----------: | :-----: |
| **ResNet101 (ImageNet)**   |  **87.00%**   | ± 2.74% |
| **DenseNet161 (ImageNet)** |  **91.00%**   | ± 3.79% |
| **DenseNet161 (BACH)**     |  **92.00%**   | ± 4.81% |

<small>Table 2: Model Performance Summary (Mean ± Std Dev)</small>

#### Analysis of Individual Model Results
The tables 1 and 2 break down the performance of each individual member of the ensemble on the hold-out set.

Evident are significant performance fluctuation across folds (e.g. ResNet101 ranges from 82.5% to 90.0%), which signalises that the model's performance is highly dependent on the specific data split. The DenseNet161 (BACH) configuration achieved the highest single-model accuracy (97.5%) but also the highest standard deviation (±4.81%). This indicates high peak performance but lower stability compared to the models that were normalised on ImageNet.
</div>
</div>


<center>
<img src="final_ensemble_confusion_matrix.png" alt="Confusion Matrices per Model: All five folds aggregated" width="60%"/>
<center>
<small>Figure 6: Aggregated Confusion Matrices per Model in the Final Ensemble Evaluation.</small>



#### Classification Report

| Class            | Precision | Recall | F1-Score | Support |
| :--------------- | :-------: | :----: | :------: | :-----: |
| **Benign**       |   1.00    |  1.00  |   1.00   |   10    |
| **InSitu**       |   0.91    |  1.00  |   0.95   |   10    |
| **Invasive**     |   0.90    |  0.90  |   0.90   |   10    |
| **Normal**       |   1.00    |  0.90  |   0.95   |   10    |
| **Accuracy**     |           |        | **0.95** | **40**  |
| **Macro Avg**    |   0.95    |  0.95  |   0.95   |   40    |
| **Weighted Avg** |   0.95    |  0.95  |   0.95   |   40    |

**Grand Ensemble Accuracy (15 models): 95.00%**

TODO 
- **Ensemble Efficacy:** Most importantly, while the average individual model accuracy hovered around 87-92%, the **Grand Ensemble achieved 95.00% accuracy**. This demonstrates that the errors made by individual models were largely uncorrelated, allowing the soft voting mechanism to correct them.


## Conclusion